# Climate Change Data Visualization
### Introductory Data Visualization Lab — 2 Hours

---

**Core Question:** *How much has the Earth warmed over the last 100 years, and why?*

**Outline:**
| # | Section | Time |
|---|---------|------|
| 0 | Introduction & Setup | 10 min |
| 1 | Data Loading & Exploration | 20 min |
| 2 | Univariate Visualization — Line Chart | 25 min |
| 3 | Multivariate Visualization — Dual Axis & Scatter | 30 min |
| 4 | Interactive Visualization with Plotly | 25 min |
| 5 | Storytelling & Wrap-up | 10 min |

---
> **Data Sources:**
> - [NASA GISS Surface Temperature Analysis (GISTEMP)](https://data.giss.nasa.gov/gistemp/)
> - [Our World in Data — CO₂ Emissions](https://ourworldindata.org/co2-emissions)
> - [NOAA Global Monitoring Laboratory — CO₂ Trends](https://gml.noaa.gov/ccgg/trends/)

---
## Section 0 — Introduction & Setup (10 min)

### Why visualize climate data?

Raw numbers in a table rarely tell a story on their own. Visualization allows us to:
- Spot **trends** and **turning points** instantly
- Communicate findings to a **non-technical audience**
- Reveal **correlations** that are invisible in raw data

Let's install and import everything we need.

In [ ]:
# Install required libraries (run once)
# !pip install pandas matplotlib seaborn plotly

In [ ]:
import pandas as pd            # data manipulation and analysis (DataFrames, CSV, merging)
import numpy as np             # numerical computing — arrays, math, interpolation
import matplotlib.pyplot as plt       # static, publication-quality plotting
import matplotlib.ticker as ticker    # fine-grained control over axis tick marks
import seaborn as sns          # statistical visualization built on top of matplotlib
import plotly.express as px    # quick, high-level interactive charts
import plotly.graph_objects as go     # lower-level Plotly — full control over traces
from plotly.subplots import make_subplots  # multi-panel Plotly layouts
import warnings
warnings.filterwarnings('ignore')     # suppress minor deprecation warnings

# Global matplotlib style — applied to every plt chart in this notebook
plt.rcParams.update({
    'figure.facecolor': '#f9f9f9',   # light gray page background
    'axes.facecolor':   '#f9f9f9',   # same for the plot area itself
    'axes.spines.top':   False,      # remove top border (cleaner look)
    'axes.spines.right': False,      # remove right border
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 14,
    'axes.labelsize': 11
})

print('✓ All libraries loaded successfully!')

---
## Section 1 — Data Loading & Exploration (20 min)

**Goal:** Understand the structure of the dataset before we visualize anything.

> **Rule of thumb:** Never visualize data you haven't explored. Always check for missing values, data types, and ranges first.

We will **download** real datasets directly from their official sources:
- **NASA GISS** global surface temperature anomalies (1880–present)
- **NOAA GML** Mauna Loa CO₂ concentrations (1959–present; ice core estimates for 1880–1958)

In [1]:
# Download Real Climate Data
# Sources: NASA GISS (temperature) · NOAA GML (CO₂) · ice core proxies (pre-1959 CO₂)

# === 1. NASA GISS Global Temperature Anomaly (1880–present) ===
giss_url = 'https://data.giss.nasa.gov/gistemp/tabledata_v4/GLB.Ts+dSST.csv'
temp_df = pd.read_csv(giss_url, skiprows=1, na_values='***')
temp_df = (temp_df[['Year', 'J-D']]
           .rename(columns={'J-D': 'Anomaly_C'})
           .dropna()
           .astype({'Year': int, 'Anomaly_C': float}))
print(f'✓ Temperature : {len(temp_df)} years ({temp_df.Year.min()}–{temp_df.Year.max()})')

# === 2. NOAA Mauna Loa CO₂ Annual Mean (1959–present) ===
co2_url = 'https://gml.noaa.gov/webdata/ccgg/trends/co2/co2_annmean_mlo.csv'
co2_df = pd.read_csv(co2_url, comment='#')
co2_df.columns = co2_df.columns.str.strip()
co2_df = (co2_df.rename(columns={'year': 'Year', 'mean': 'CO2_ppm'})[['Year', 'CO2_ppm']]
          .astype({'Year': int, 'CO2_ppm': float}))
print(f'✓ CO₂ (direct): {len(co2_df)} years ({co2_df.Year.min()}–{co2_df.Year.max()})')

# === 3. Merge — temperature is the base (1880–present) ===
df = temp_df.merge(co2_df, on='Year', how='left')

# Pre-1959 CO₂: interpolated from Law Dome ice core proxies
# (~295 ppm in 1880, known 315.97 ppm in 1959)
mask = df['CO2_ppm'].isna()
first_co2 = co2_df.iloc[0]['CO2_ppm']   # 1959 Mauna Loa value
df.loc[mask, 'CO2_ppm'] = np.round(
    np.interp(df.loc[mask, 'Year'], [1880, 1959], [295.0, first_co2]), 1
)
print(f'  (CO₂ 1880–1958: estimated from ice core proxies — {mask.sum()} years)')

# === 4. Sea Level (mm, relative to 1990 baseline) ===
# CSIRO/Church & White 2011 trend approximation (~1.2 mm/yr pre-1990, ~3.5 mm/yr after)
df['SeaLevel_mm'] = np.round(
    np.where(df['Year'] < 1990,
             (df['Year'] - 1880) * 1.2 - 132,
             (df['Year'] - 1990) * 3.5), 1
)

df = df.round({'Anomaly_C': 3, 'CO2_ppm': 1}).reset_index(drop=True)
print(f'\nDataset ready: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(10)

NameError: name 'pd' is not defined

In [ ]:
# Basic Exploratory Analysis — always do this BEFORE charting!

print('=== Descriptive Statistics ===')
display(df.describe().round(2))   # mean, std, min, 25th/50th/75th percentile, max per column

print('\n=== Missing Values ===')
print(df.isnull().sum())          # count NaN per column — 0 means no gaps

print('\n=== Data Types ===')
print(df.dtypes)                  # confirms Year=int64, Anomaly_C=float64, etc.

In [ ]:
# Discussion Questions

# 1. What does a temperature anomaly of +0.5°C actually mean?
# → It's the difference from the 1951–1980 long-term average (NASA GISS baseline)
#   Positive = warmer than that average; negative = cooler

# 2. What was the pre-industrial CO₂ level, and what is it today?
# → ~280 ppm before 1850; over 420 ppm as of 2024 (direct Mauna Loa measurement)

# 3. What years show the most dramatic changes in the data?
yr_min = df['Year'].min()   # first year in the real dataset (should be 1880)
yr_max = df['Year'].max()   # most recent year downloaded

print('Max temperature anomaly year:', df.loc[df['Anomaly_C'].idxmax(), 'Year'])
print('Max CO₂ year               :', df.loc[df['CO2_ppm'].idxmax(), 'Year'])
print(f'Anomaly in {yr_min}        :', df.loc[df['Year'] == yr_min, 'Anomaly_C'].values[0])
print(f'Anomaly in {yr_max}        :', df.loc[df['Year'] == yr_max, 'Anomaly_C'].values[0])

---
## Section 2 — Univariate Visualization: Line Chart (25 min)

**Goal:** Show the temperature anomaly trend over time — one variable, clearly told.

> **Design principle:** A good chart has one clear message. Ask yourself: *"What is the single thing I want viewers to take away?"*

In [ ]:
# Basic Line Chart
yr_min, yr_max = df['Year'].min(), df['Year'].max()

fig, ax = plt.subplots(figsize=(13, 5))   # width=13", height=5" — wide format suits time series

ax.plot(df['Year'], df['Anomaly_C'],
        color='#d62728',    # muted red — intuitively signals heat/warmth
        linewidth=1.2,      # thin enough to show year-to-year variability
        alpha=0.6,          # semi-transparent so overlapping features stay visible
        label='Annual Anomaly')

# axhline: horizontal reference line at y=0 (the 1951–1980 average baseline)
ax.axhline(0, color='#555555', linestyle='--', linewidth=1, label='Baseline (1951–1980 avg)')

ax.set_title(f'Global Surface Temperature Anomaly ({yr_min}–{yr_max})',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Year')
ax.set_ylabel('Temperature Anomaly (°C)')
ax.legend(frameon=False)   # frameon=False removes the legend box border

plt.tight_layout()   # auto-adjusts spacing so labels aren't clipped
plt.show()

In [ ]:
# Enhanced: Add 10-Year Rolling Average + Shaded Regions
yr_min, yr_max = df['Year'].min(), df['Year'].max()

# rolling(window=10): compute a moving average over 10 consecutive years
# center=True: the window is centered on each year (5 before, 5 after)
#              smooths short-term noise to reveal the long-term trend
df['Rolling10'] = df['Anomaly_C'].rolling(window=10, center=True).mean()

fig, ax = plt.subplots(figsize=(13, 5))

# fill_between: shade the area between the anomaly line and y=0
# where=: Boolean mask — only shade where the condition is True
ax.fill_between(df['Year'], df['Anomaly_C'], 0,
                where=df['Anomaly_C'] >= 0,
                color='#d62728', alpha=0.25, label='Warmer than baseline')
ax.fill_between(df['Year'], df['Anomaly_C'], 0,
                where=df['Anomaly_C'] < 0,
                color='#1f77b4', alpha=0.25, label='Cooler than baseline')

# Raw annual values — faint gray so they don't compete with the rolling mean
ax.plot(df['Year'], df['Anomaly_C'],
        color='#888888', linewidth=0.8, alpha=0.5)

# Rolling mean — thick line highlights the underlying multi-decade trend
ax.plot(df['Year'], df['Rolling10'],
        color='#d62728', linewidth=2.5, label='10-Year Rolling Mean')

ax.axhline(0, color='#333333', linestyle='--', linewidth=1)

# axvline: vertical reference line marking a key historical turning point
ax.axvline(1980, color='orange', linestyle=':', linewidth=1.5)
ax.text(1981, df['Anomaly_C'].min() + 0.05,   # y: just above data minimum
        'Rapid warming\nbegins (~1980)', fontsize=9, color='darkorange')

ax.set_title(f'Global Surface Temperature Anomaly ({yr_min}–{yr_max})',
             fontsize=15, fontweight='bold', pad=15)
ax.set_xlabel('Year')
ax.set_ylabel('Temperature Anomaly (°C)')
ax.legend(frameon=False, loc='upper left')

plt.tight_layout()
plt.show()

print('\n💬 Discussion: What changed around 1980? What events coincide with this turning point?')

In [ ]:
# Bonus: Box Plot — Anomaly Distribution by Era

# pd.cut: bin a continuous numeric column (Year) into labeled categorical eras
# bins: the boundary values (left-exclusive, right-inclusive by default)
# labels: the name assigned to each interval
df['Era'] = pd.cut(df['Year'],
                   bins=[1879, 1919, 1949, 1979, 2009, df['Year'].max() + 1],
                   labels=['1880s–1919', '1920s–1949', '1950s–1979',
                           '1980s–2009', '2010s–present'])

fig, ax = plt.subplots(figsize=(10, 5))

# coolwarm: blue (cool) → red (warm) — palette choice reinforces the temperature narrative
palette = sns.color_palette('coolwarm', 5)

# Box anatomy: center line = median; box = 25th–75th percentile (IQR);
# whiskers = 1.5× IQR beyond the box; dots beyond whiskers = outliers
sns.boxplot(data=df, x='Era', y='Anomaly_C',
            palette=palette, ax=ax, width=0.5)

ax.axhline(0, color='gray', linestyle='--', linewidth=1)   # baseline reference
ax.set_title('Temperature Anomaly Distribution by Era', fontsize=14, fontweight='bold')
ax.set_xlabel('Era')
ax.set_ylabel('Temperature Anomaly (°C)')

plt.tight_layout()
plt.show()

print('\n💬 Discussion: How does the median anomaly shift across eras?')

---
## Section 3 — Multivariate Visualization (30 min)

**Goal:** Explore the relationship between CO₂ concentration and temperature anomaly.

> **Important:** *Correlation ≠ Causation.* A strong visual correlation is a starting point for inquiry, not proof.

In [ ]:
# Dual-Axis Chart: Temperature + CO₂ over Time
# Use a dual y-axis when two variables share the same x-axis but have very different scales
yr_min, yr_max = df['Year'].min(), df['Year'].max()

fig, ax1 = plt.subplots(figsize=(13, 5))

color_temp = '#d62728'   # red for temperature — warm connotation
color_co2  = '#1a6eb5'   # blue for CO₂ — neutral / industrial connotation

ax1.plot(df['Year'], df['Anomaly_C'], color=color_temp,
         linewidth=1.5, label='Temperature Anomaly (°C)')
ax1.set_xlabel('Year')
ax1.set_ylabel('Temperature Anomaly (°C)', color=color_temp)
ax1.tick_params(axis='y', labelcolor=color_temp)   # match tick color to the line
ax1.axhline(0, color=color_temp, linestyle=':', alpha=0.4)

# twinx(): creates a second y-axis that shares the same x-axis as ax1
# Each axis can have an independent scale — essential here (°C vs ppm)
ax2 = ax1.twinx()
ax2.plot(df['Year'], df['CO2_ppm'], color=color_co2,
         linewidth=1.5, linestyle='--', label='CO₂ Concentration (ppm)')
ax2.set_ylabel('CO₂ Concentration (ppm)', color=color_co2)
ax2.tick_params(axis='y', labelcolor=color_co2)

# get_legend_handles_labels(): returns the line objects and their labels
# Merge both axes' handles into a single legend on ax1
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, frameon=False, loc='upper left')

ax1.set_title(f'Global Temperature Anomaly vs. CO₂ Concentration ({yr_min}–{yr_max})',
              fontsize=14, fontweight='bold', pad=15)

plt.tight_layout()
plt.show()

print('\n💬 Discussion: Do the two lines move together? When do they diverge or converge?')

In [ ]:
# Scatter Plot: CO₂ vs Temperature Anomaly (colored by Year)
# Goal: reveal whether CO₂ level predicts temperature across all years

fig, ax = plt.subplots(figsize=(9, 6))

# c=df['Year']: encodes time as color — earlier years lighter, recent years darker red
# cmap='YlOrRd': yellow → orange → red colormap (perceptually ordered)
# s=30: marker size in points²; edgecolors='none': no border around dots
sc = ax.scatter(df['CO2_ppm'], df['Anomaly_C'],
                c=df['Year'], cmap='YlOrRd',
                s=30, alpha=0.8, edgecolors='none')

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Year', fontsize=10)

# np.polyfit(x, y, deg): fits a polynomial of degree `deg` to the data
# deg=1 → straight line; returns [slope, intercept]
z = np.polyfit(df['CO2_ppm'], df['Anomaly_C'], 1)

# np.poly1d: wraps the coefficients into a callable function p(x)
p = np.poly1d(z)

x_line = np.linspace(df['CO2_ppm'].min(), df['CO2_ppm'].max(), 200)
ax.plot(x_line, p(x_line), 'k--', linewidth=1.5, label='Trend line')

# Pearson r: linear correlation coefficient, ranges from -1 to +1
# r ≈ +1 → strong positive relationship; r ≈ 0 → no linear relationship
corr = df['CO2_ppm'].corr(df['Anomaly_C'])

# transform=ax.transAxes: x/y coordinates are in axis-fraction space (0–1),
# not data units — so the label stays in the same corner regardless of data range
ax.text(0.05, 0.92, f'Pearson r = {corr:.3f}',
        transform=ax.transAxes, fontsize=11,
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

ax.set_title('CO₂ Concentration vs. Temperature Anomaly', fontsize=14, fontweight='bold')
ax.set_xlabel('CO₂ Concentration (ppm)')
ax.set_ylabel('Temperature Anomaly (°C)')
ax.legend(frameon=False)

plt.tight_layout()
plt.show()

print(f'\n📊 Pearson Correlation Coefficient: {corr:.3f}')
print('💬 Discussion: What does a correlation of ~0.97 tell us? What does it NOT tell us?')

In [ ]:
# Heatmap: Average Climate Indicators by Decade

# Floor-divide Year by 10, then multiply — groups every year into its decade
# e.g. 1987 → (1987 // 10) * 10 → 1980
df['Decade'] = (df['Year'] // 10) * 10

# groupby + mean: compute one representative value per decade for each indicator
heatmap_data = df.groupby('Decade')[['Anomaly_C', 'CO2_ppm', 'SeaLevel_mm']].mean()

fig, ax = plt.subplots(figsize=(10, 4))

# sns.heatmap parameters:
#   .T          — transpose so decades are columns, indicators are rows
#   cmap        — 'RdYlBu_r': reversed Red-Yellow-Blue (red = high, blue = low)
#   annot=True  — print the numeric value inside each cell
#   fmt='.1f'   — format annotation as 1 decimal place
#   linewidths  — thin white grid lines between cells for readability
sns.heatmap(
    heatmap_data.T,
    cmap='RdYlBu_r',
    annot=True, fmt='.1f',
    linewidths=0.5,
    ax=ax,
    cbar_kws={'label': 'Value'}
)

ax.set_title('Climate Indicators by Decade (Decade Averages)', fontsize=13, fontweight='bold')
ax.set_xlabel('Decade')
ax.set_yticklabels(['Temp Anomaly (°C)', 'CO₂ (ppm)', 'Sea Level (mm)'], rotation=0)

plt.tight_layout()
plt.show()

---
## Section 4 — Interactive Visualization with Plotly (25 min)

**Goal:** Create charts users can explore — zoom, hover, filter.

> **When to use interactive charts?** 
> - Presentations where the audience asks *"What about...?"* 
> - Dashboards where users explore data themselves 
> - Large datasets where zooming in reveals detail

In [ ]:
# Interactive Line Chart
yr_min, yr_max = df['Year'].min(), df['Year'].max()

# Ensure Rolling10 exists (computed in the enhanced line chart cell)
if 'Rolling10' not in df.columns:
    df['Rolling10'] = df['Anomaly_C'].rolling(window=10, center=True).mean()

# px.line: Plotly Express wrapper — produces interactive charts with minimal code
# labels: renames DataFrame columns to human-readable axis titles
fig = px.line(
    df, x='Year', y='Anomaly_C',
    title=f'Global Temperature Anomaly ({yr_min}–{yr_max}) — Interactive',
    labels={'Anomaly_C': 'Temperature Anomaly (°C)', 'Year': 'Year'},
    color_discrete_sequence=['#d62728']
)

# add_scatter: overlay an additional trace on the existing figure
# mode='lines': draw as a line (not markers or text)
fig.add_scatter(x=df['Year'], y=df['Rolling10'],
                mode='lines', name='10-Year Rolling Mean',
                line=dict(color='darkred', width=3))

# add_hline: horizontal reference line spanning the full x range
fig.add_hline(y=0, line_dash='dash', line_color='gray',
              annotation_text='Baseline (1951–1980 avg)',
              annotation_position='bottom right')

fig.update_layout(
    plot_bgcolor='#f9f9f9',
    paper_bgcolor='white',
    hovermode='x unified',   # show all series values for the same x on hover
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)

fig.show()
print('\n🖱️ Try: zoom into 1980–present by clicking and dragging on the chart!')

In [ ]:
# Interactive Scatter Plot: CO₂ vs Temperature

# hover_data: controls what appears in the tooltip when the user hovers over a point
#   True       → show the column value with default formatting
#   ':.1f'     → show with 1 decimal place
#   ':.3f'     → show with 3 decimal places
fig = px.scatter(
    df, x='CO2_ppm', y='Anomaly_C',
    color='Year',                        # encodes time as a continuous color gradient
    color_continuous_scale='YlOrRd',     # yellow (early) → red (recent)
    hover_data={'Year': True, 'CO2_ppm': ':.1f', 'Anomaly_C': ':.3f'},
    title='CO₂ Concentration vs. Temperature Anomaly (hover for details)',
    labels={'CO2_ppm': 'CO₂ (ppm)', 'Anomaly_C': 'Temperature Anomaly (°C)'}
)

# trendline='ols': fits an Ordinary Least Squares regression line automatically
# px.scatter(...).data returns a tuple of traces: [0] = scatter points, [1] = trendline
# We add only the trendline (.data[1]) to our existing figure
fig.add_traces(
    px.scatter(df, x='CO2_ppm', y='Anomaly_C', trendline='ols').data[1]
)

fig.update_layout(
    plot_bgcolor='#f9f9f9',
    paper_bgcolor='white'
)

fig.show()
print('\n🖱️ Try: hover over any point to see the exact year, CO₂ level, and temperature anomaly!')

In [ ]:
# Mini Dashboard: 3 Indicators Side by Side
yr_min, yr_max = df['Year'].min(), df['Year'].max()

# make_subplots: creates a grid of linked Plotly chart panels
# rows=1, cols=3: one row, three columns
# shared_xaxes=False: each panel has its own independent x-axis scale
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=(
        '🌡️ Temperature Anomaly',
        '🏭 CO₂ Concentration',
        '🌊 Sea Level Change'
    ),
    shared_xaxes=False
)

# go.Scatter: lower-level Plotly object (more control than px.line)
# fill='tozeroy': shade the area between the line and y=0
# fillcolor: semi-transparent fill using rgba(R, G, B, alpha)

# Chart 1: Temperature
fig.add_trace(
    go.Scatter(x=df['Year'], y=df['Anomaly_C'],
               fill='tozeroy',
               fillcolor='rgba(214,39,40,0.15)',   # translucent red fill
               line=dict(color='#d62728', width=1.5),
               name='Temp Anomaly'),
    row=1, col=1   # position in the subplot grid
)

# Chart 2: CO₂
fig.add_trace(
    go.Scatter(x=df['Year'], y=df['CO2_ppm'],
               line=dict(color='#1a6eb5', width=1.5),
               name='CO₂ (ppm)'),
    row=1, col=2
)

# Chart 3: Sea Level
fig.add_trace(
    go.Scatter(x=df['Year'], y=df['SeaLevel_mm'],
               fill='tozeroy',
               fillcolor='rgba(23,190,207,0.15)',   # translucent teal fill
               line=dict(color='#17becf', width=1.5),
               name='Sea Level (mm)'),
    row=1, col=3
)

fig.update_layout(
    title_text=f'Climate Change — Three Key Indicators ({yr_min}–{yr_max})',
    title_font_size=16,
    showlegend=False,      # subplot titles serve as labels — no separate legend needed
    plot_bgcolor='#f9f9f9',
    paper_bgcolor='white',
    height=380
)

fig.show()

---
## Section 5 — Storytelling & Wrap-Up (10 min)

**Goal:** Turn three charts into a coherent data story.

### Your Mini Storyboard

| Slide | Chart | Message |
|-------|-------|---------|
| 1 — Hook | Line chart (Section 2) | *"The Earth has warmed by over 1°C since 1880 — most of it after 1980"* |
| 2 — Evidence | Dual-axis / Scatter (Section 3) | *"CO₂ concentration and temperature rise are almost perfectly correlated (r ≈ 0.97)"* |
| 3 — Insight | Interactive dashboard (Section 4) | *"All three indicators — temperature, CO₂, sea level — are accelerating together"* |

---

### Peer Feedback Checklist

After viewing a classmate's charts, evaluate them using these criteria:

- [ ] **Clarity** — Is the main message obvious within 5 seconds?
- [ ] **Color** — Does the color palette reinforce the message (e.g., red = warm)?
- [ ] **Labels** — Are axes, titles, and units clearly labeled?
- [ ] **Source** — Is the data source cited?
- [ ] **Simplicity** — Is anything in the chart unnecessary or distracting?

In [ ]:
# Final Summary Chart — presentation-ready, saved to file
yr_min, yr_max = df['Year'].min(), df['Year'].max()

# Ensure Rolling10 exists (computed in line-chart-enhanced)
if 'Rolling10' not in df.columns:
    df['Rolling10'] = df['Anomaly_C'].rolling(window=10, center=True).mean()

fig, ax = plt.subplots(figsize=(13, 5))

# Shade warmer/cooler periods relative to baseline
ax.fill_between(df['Year'], df['Anomaly_C'], 0,
                where=df['Anomaly_C'] >= 0, color='#d62728', alpha=0.3)
ax.fill_between(df['Year'], df['Anomaly_C'], 0,
                where=df['Anomaly_C'] < 0, color='#1f77b4', alpha=0.3)

# Thick rolling mean line carries the main trend message
ax.plot(df['Year'], df['Rolling10'], color='#d62728', linewidth=2.5, label='10-Year Mean')
ax.axhline(0, color='#444444', linestyle='--', linewidth=1)

# annotate(): add text with an optional arrow pointing to a data coordinate
# xy: the tip of the arrow (data coordinates); xytext: where the label sits
ax.annotate('Industrial\nRevolution', xy=(1880, -0.3), fontsize=8, color='gray')
ax.annotate('Post-WWII\nGrowth',      xy=(1950, -0.3), fontsize=8, color='gray')
ax.annotate('Rapid warming\naccelerates', xy=(1990, 0.35),
            fontsize=9, color='#d62728',
            arrowprops=dict(arrowstyle='->', color='#d62728'),
            xytext=(1970, 0.65))

ax.set_title('One Hundred and Forty Years of Global Warming',
             fontsize=16, fontweight='bold', pad=15)
ax.set_xlabel('Year', fontsize=11)
ax.set_ylabel('Temperature Anomaly (°C)', fontsize=11)
ax.set_xlim(yr_min, yr_max)   # dynamic — adapts as new data arrives
ax.legend(frameon=False)

# figtext: place text relative to the figure (not the axes) — useful for source credits
plt.figtext(0.99, 0.01,
            'Source: NASA GISS Surface Temperature Analysis (GISTEMP v4)',
            ha='right', fontsize=8, color='gray')

plt.tight_layout()
plt.savefig('climate_summary_chart.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Chart saved as climate_summary_chart.png')
print('\n📌 Lab complete! Key takeaways:')
print('  1. Always explore your data before visualizing')
print('  2. Design choices (color, annotation) shape the story')
print('  3. Correlation is not causation — but it is a clue')
print('  4. Interactive charts invite exploration; static charts make arguments')

---
## Further Resources

| Resource | Link |
|----------|------|
| NASA GISS Temperature Data | https://data.giss.nasa.gov/gistemp/ |
| Our World in Data — CO₂ | https://ourworldindata.org/co2-emissions |
| Matplotlib Gallery | https://matplotlib.org/stable/gallery/ |
| Plotly Python Docs | https://plotly.com/python/ |
| Seaborn Tutorial | https://seaborn.pydata.org/tutorial.html |
| Data Visualization Best Practices | https://clauswilke.com/dataviz/ |

---
*End of Lab — Climate Change Data Visualization (2 Hours)*